# Task 1 — ML Classification Project: Customer Churn Prediction
**Alfido Tech Internship**  
Dataset: Telco Customer Churn (Kaggle)  
Goal: Compare Logistic Regression vs Random Forest — report Accuracy, Precision, Recall, F1, ROC-AUC

## Cell 1 — Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, classification_report,
    confusion_matrix, RocCurveDisplay, ConfusionMatrixDisplay
)

import os
os.makedirs('../outputs', exist_ok=True)

print('All libraries imported successfully')

## Cell 2 — Load Dataset

In [ ]:
# Download from: https://www.kaggle.com/datasets/blastchar/telco-customer-churn
# Place the CSV in the data/ folder as telco_churn.csv

df = pd.read_csv('../data/telco_churn.csv')

print('Shape:', df.shape)
print('\nColumn names:')
print(df.columns.tolist())
print('\nFirst 5 rows:')
df.head()

## Cell 3 — Data Cleaning & Preprocessing

In [ ]:
# Drop customerID — not a useful feature
df.drop('customerID', axis=1, inplace=True)

# TotalCharges is stored as string — convert to float
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

# Check for missing values
print('Missing values per column:')
print(df.isnull().sum()[df.isnull().sum() > 0])

# Fill missing TotalCharges with median
df['TotalCharges'].fillna(df['TotalCharges'].median(), inplace=True)

# Encode target: Yes -> 1, No -> 0
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

print('\nChurn value counts:')
print(df['Churn'].value_counts())
print(f'\nClass balance: {df["Churn"].mean()*100:.1f}% churned')

## Cell 4 — Exploratory Data Analysis (EDA)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Exploratory Data Analysis — Telco Churn', fontsize=16, fontweight='bold')

# Plot 1: Histogram — Tenure distribution
axes[0, 0].hist(df['tenure'], bins=30, color='#7F77DD', edgecolor='white')
axes[0, 0].set_title('Tenure Distribution (months)')
axes[0, 0].set_xlabel('Tenure')
axes[0, 0].set_ylabel('Count')

# Plot 2: Boxplot — Monthly Charges by Churn
churn_labels = df['Churn'].map({0: 'No Churn', 1: 'Churned'})
df_plot = df.copy()
df_plot['Churn_Label'] = churn_labels
sns.boxplot(data=df_plot, x='Churn_Label', y='MonthlyCharges',
            ax=axes[0, 1], palette={'No Churn': '#5DCAA5', 'Churned': '#D85A30'})
axes[0, 1].set_title('Monthly Charges by Churn Status')

# Plot 3: Scatter — Tenure vs Total Charges
colors = df['Churn'].map({0: '#5DCAA5', 1: '#D85A30'})
axes[1, 0].scatter(df['tenure'], df['TotalCharges'], c=colors, alpha=0.4, s=15)
axes[1, 0].set_title('Tenure vs Total Charges')
axes[1, 0].set_xlabel('Tenure (months)')
axes[1, 0].set_ylabel('Total Charges')
from matplotlib.patches import Patch
legend_elements = [Patch(color='#5DCAA5', label='No Churn'), Patch(color='#D85A30', label='Churned')]
axes[1, 0].legend(handles=legend_elements)

# Plot 4: Bar — Churn rate by Contract type
churn_by_contract = df.groupby('Contract')['Churn'].mean().sort_values(ascending=False)
axes[1, 1].bar(churn_by_contract.index, churn_by_contract.values,
               color=['#D85A30', '#F0997B', '#5DCAA5'])
axes[1, 1].set_title('Churn Rate by Contract Type')
axes[1, 1].set_ylabel('Churn Rate')
axes[1, 1].set_xlabel('Contract Type')
for i, v in enumerate(churn_by_contract.values):
    axes[1, 1].text(i, v + 0.01, f'{v:.1%}', ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('../outputs/eda_plots.png', dpi=150, bbox_inches='tight')
plt.show()
print('EDA plots saved to outputs/eda_plots.png')

## Cell 5 — Feature Engineering & Encoding

In [ ]:
df_model = df.copy()

# Encode all object columns with LabelEncoder
le = LabelEncoder()
cat_cols = df_model.select_dtypes(include='object').columns.tolist()
print('Encoding columns:', cat_cols)

for col in cat_cols:
    df_model[col] = le.fit_transform(df_model[col])

# Feature matrix and target
X = df_model.drop('Churn', axis=1)
y = df_model['Churn']

feature_names = X.columns.tolist()
print(f'\nFeatures ({len(feature_names)}):', feature_names)
print('Target distribution:\n', y.value_counts())

## Cell 6 — Train/Test Split & Scaling

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Training set: {X_train.shape[0]} samples')
print(f'Test set:     {X_test.shape[0]} samples')
print(f'Train churn rate: {y_train.mean():.3f}')
print(f'Test churn rate:  {y_test.mean():.3f}')

# Scale features
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print('\nScaling done. Feature means (should be ~0):', X_train_sc.mean(axis=0)[:3].round(3))

## Cell 7 — Model 1: Logistic Regression

In [ ]:
lr = LogisticRegression(max_iter=1000, C=1.0, random_state=42)
lr.fit(X_train_sc, y_train)

y_pred_lr   = lr.predict(X_test_sc)
y_prob_lr   = lr.predict_proba(X_test_sc)[:, 1]

print('=== Logistic Regression Results ===')
print(f'Accuracy:  {accuracy_score(y_test, y_pred_lr):.4f}')
print(f'Precision: {precision_score(y_test, y_pred_lr):.4f}')
print(f'Recall:    {recall_score(y_test, y_pred_lr):.4f}')
print(f'F1 Score:  {f1_score(y_test, y_pred_lr):.4f}')
print(f'ROC-AUC:   {roc_auc_score(y_test, y_prob_lr):.4f}')
print('\nClassification Report:')
print(classification_report(y_test, y_pred_lr, target_names=['No Churn', 'Churn']))

## Cell 8 — Model 2: Random Forest

In [ ]:
rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    min_samples_split=5,
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train_sc, y_train)

y_pred_rf   = rf.predict(X_test_sc)
y_prob_rf   = rf.predict_proba(X_test_sc)[:, 1]

print('=== Random Forest Results ===')
print(f'Accuracy:  {accuracy_score(y_test, y_pred_rf):.4f}')
print(f'Precision: {precision_score(y_test, y_pred_rf):.4f}')
print(f'Recall:    {recall_score(y_test, y_pred_rf):.4f}')
print(f'F1 Score:  {f1_score(y_test, y_pred_rf):.4f}')
print(f'ROC-AUC:   {roc_auc_score(y_test, y_prob_rf):.4f}')
print('\nClassification Report:')
print(classification_report(y_test, y_pred_rf, target_names=['No Churn', 'Churn']))

## Cell 9 — Cross Validation (5-Fold)

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

lr_cv  = cross_val_score(lr, X_train_sc, y_train, cv=cv, scoring='f1')
rf_cv  = cross_val_score(rf, X_train_sc, y_train, cv=cv, scoring='f1')

print('=== 5-Fold Cross Validation (F1 Score) ===')
print(f'Logistic Regression: {lr_cv.mean():.3f} ± {lr_cv.std():.3f}')
print(f'Random Forest:       {rf_cv.mean():.3f} ± {rf_cv.std():.3f}')

print('\nFold-by-fold:')
for i, (lr_s, rf_s) in enumerate(zip(lr_cv, rf_cv), 1):
    print(f'  Fold {i}: LR={lr_s:.3f}  RF={rf_s:.3f}')

## Cell 10 — ROC Curve Comparison

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

RocCurveDisplay.from_predictions(
    y_test, y_prob_lr, name=f'Logistic Regression (AUC={roc_auc_score(y_test,y_prob_lr):.3f})',
    ax=ax, color='#7F77DD'
)
RocCurveDisplay.from_predictions(
    y_test, y_prob_rf, name=f'Random Forest (AUC={roc_auc_score(y_test,y_prob_rf):.3f})',
    ax=ax, color='#1D9E75'
)
ax.plot([0, 1], [0, 1], 'k--', label='Random classifier')
ax.set_title('ROC Curve — Logistic Regression vs Random Forest', fontsize=13)
ax.legend(loc='lower right')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')

plt.tight_layout()
plt.savefig('../outputs/roc_curve.png', dpi=150, bbox_inches='tight')
plt.show()
print('ROC curve saved to outputs/roc_curve.png')

## Cell 11 — Confusion Matrices

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred_lr, display_labels=['No Churn', 'Churn'],
    ax=ax1, cmap='Purples'
)
ax1.set_title('Logistic Regression')

ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred_rf, display_labels=['No Churn', 'Churn'],
    ax=ax2, cmap='Greens'
)
ax2.set_title('Random Forest')

plt.suptitle('Confusion Matrices', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print('Confusion matrix saved to outputs/confusion_matrix.png')

## Cell 12 — Feature Importance (Random Forest)

In [ ]:
importances = rf.feature_importances_
indices     = np.argsort(importances)[::-1]
top_n       = 10

fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(
    [feature_names[i] for i in indices[:top_n]][::-1],
    importances[indices[:top_n]][::-1],
    color='#7F77DD'
)
ax.set_title('Top 10 Feature Importances — Random Forest')
ax.set_xlabel('Importance Score')
plt.tight_layout()
plt.savefig('../outputs/feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print('Feature importance plot saved to outputs/feature_importance.png')

## Cell 13 — Final Comparison Summary

In [ ]:
results = pd.DataFrame({
    'Model': ['Logistic Regression', 'Random Forest'],
    'Accuracy':  [accuracy_score(y_test, y_pred_lr),  accuracy_score(y_test, y_pred_rf)],
    'Precision': [precision_score(y_test, y_pred_lr), precision_score(y_test, y_pred_rf)],
    'Recall':    [recall_score(y_test, y_pred_lr),    recall_score(y_test, y_pred_rf)],
    'F1 Score':  [f1_score(y_test, y_pred_lr),        f1_score(y_test, y_pred_rf)],
    'ROC-AUC':   [roc_auc_score(y_test, y_prob_lr),   roc_auc_score(y_test, y_prob_rf)],
})
results.set_index('Model', inplace=True)
results = results.round(4)

print('FINAL MODEL COMPARISON')
print(results.to_string())

winner = results['ROC-AUC'].idxmax()
print(f'\nWinner by ROC-AUC: {winner}')

results

## Cell 14 — Save Model for Task 3 (Deployment)

In [ ]:
import joblib

joblib.dump(rf,     '../model.pkl')
joblib.dump(scaler, '../scaler.pkl')
joblib.dump(feature_names, '../feature_names.pkl')

print('Saved:')
print('  model.pkl        — trained Random Forest')
print('  scaler.pkl       — fitted StandardScaler')
print('  feature_names.pkl — list of feature names')
print('\nThese files are reused in Task 3 (deployment) and Task 4 (SHAP analysis)')